In [2]:
import re
import requests
import pandas as pd


BASE_URL = "https://dati.arpacampania.it/api/3/action"

DATASETS = {
    "nrt": "dati-grezzi-orari-qualita-aria",          # dati grezzi near real-time
    "validati": "dati-rqa-giornalieri-validati",     # dati validati
}


def _ckan(action: str, payload: dict) -> dict:
    r = requests.post(
        f"{BASE_URL}/{action}",
        json=payload,
        timeout=60
    )
    r.raise_for_status()

    data = r.json()
    if not data.get("success"):
        raise RuntimeError(data.get("error", "Errore API CKAN"))

    return data["result"]


def _parse_mese_risorsa(resource: dict):
    """
    Estrae mese/anno dal nome risorsa, es. 'Dati RQA NRT grezzi 06-2026'.
    Ritorna (anno, mese).
    """
    name = resource.get("name", "")
    match = re.search(r"(\d{2})-(\d{4})", name)

    if not match:
        return None

    mese, anno = match.groups()
    return int(anno), int(mese)


def _mappa_risorse_mensili(tipo: str = "nrt") -> dict:
    """
    Ritorna una mappa:
    {
        (2026, 6): {"id": "...", "name": "..."},
        ...
    }
    """
    if tipo not in DATASETS:
        raise ValueError(f"tipo deve essere uno tra: {list(DATASETS)}")

    package = _ckan("package_show", {"id": DATASETS[tipo]})

    risorse = {}
    for r in package["resources"]:
        if not r.get("datastore_active"):
            continue

        ym = _parse_mese_risorsa(r)
        if ym:
            risorse[ym] = r

    return risorse


def _mesi_nel_periodo(data_inizio, data_fine):
    start = pd.Timestamp(data_inizio)
    end = pd.Timestamp(data_fine)

    # end è esclusivo
    if end <= start:
        raise ValueError("data_fine deve essere successiva a data_inizio")

    last_included = end - pd.Timedelta(microseconds=1)

    return pd.period_range(
        start=start.to_period("M"),
        end=last_included.to_period("M"),
        freq="M"
    )


def _scarica_records_centralina(resource_id: str, codice_centralina: str) -> list[dict]:
    """
    Scarica tutti i record della centralina da una singola risorsa mensile.
    Usa paginazione con offset.
    """
    records = []
    offset = 0
    limit = 32000

    while True:
        result = _ckan("datastore_search", {
            "resource_id": resource_id,
            "filters": {
                "Stazione": codice_centralina
            },
            "fields": [
                "Stazione",
                "Descrizione",
                "Inquinante",
                "Data_ora",
                "Valore",
                "Um"
            ],
            "limit": limit,
            "offset": offset,
            "include_total": True
        })

        batch = result["records"]
        records.extend(batch)

        if not batch:
            break

        offset += len(batch)

        if offset >= result.get("total", 0):
            break

    return records


def dati_centralina_periodo(
    codice_centralina: str,
    data_inizio,
    data_fine=None,
    tipo: str = "nrt",
    formato: str = "long"
) -> pd.DataFrame:
    """
    Restituisce un DataFrame con tutti gli inquinanti misurati da una centralina.

    Parametri:
    - codice_centralina: es. "IT0898A"
    - data_inizio: es. "2026-06-01"
    - data_fine: es. "2026-06-07"; se None, prende l'intera giornata di data_inizio
    - tipo: "nrt" oppure "validati"
    - formato:
        - "long": una riga per inquinante/orario
        - "wide": una colonna per inquinante

    Nota:
    data_fine è inclusiva se passi una data senza ora.
    """

    start = pd.Timestamp(data_inizio)

    if data_fine is None:
        end = start + pd.Timedelta(days=1)
    else:
        end = pd.Timestamp(data_fine)

        # Se data_fine è una stringa data-only, includo tutta la giornata
        if isinstance(data_fine, str) and len(data_fine.strip()) <= 10:
            end = end + pd.Timedelta(days=1)

    risorse = _mappa_risorse_mensili(tipo)
    mesi = _mesi_nel_periodo(start, end)

    frames = []

    for periodo in mesi:
        key = (periodo.year, periodo.month)

        if key not in risorse:
            continue

        resource_id = risorse[key]["id"]
        records = _scarica_records_centralina(resource_id, codice_centralina)

        if records:
            frames.append(pd.DataFrame(records))

    if not frames:
        return pd.DataFrame(columns=[
            "Stazione", "Descrizione", "Inquinante", "Data_ora", "Data_ora_dt", "Valore", "Um"
        ])

    df = pd.concat(frames, ignore_index=True)

    # Data_ora ARPAC è testo tipo "01-06-2026 00:00:00 +01:00".
    # Tolgo offset finale e interpreto come data/ora locale dichiarata nel dato.
    df["Data_ora_dt"] = (
        df["Data_ora"]
        .astype(str)
        .str.replace(r"\s+[+-]\d{2}:\d{2}$", "", regex=True)
    )

    df["Data_ora_dt"] = pd.to_datetime(
        df["Data_ora_dt"],
        format="%d-%m-%Y %H:%M:%S",
        errors="coerce"
    )

    df["Valore"] = (
        df["Valore"]
        .astype(str)
        .str.replace(",", ".", regex=False)
    )

    df["Valore"] = pd.to_numeric(df["Valore"], errors="coerce")

    df = df[
        (df["Data_ora_dt"] >= start) &
        (df["Data_ora_dt"] < end)
    ].copy()

    df = df.sort_values(["Data_ora_dt", "Inquinante"]).reset_index(drop=True)

    if formato == "wide":
        df = (
            df.pivot_table(
                index=["Data_ora_dt", "Stazione", "Descrizione"],
                columns="Inquinante",
                values="Valore",
                aggfunc="first"
            )
            .reset_index()
        )

        df.columns.name = None

    return df

In [3]:
df = dati_centralina_periodo(
    codice_centralina="IT0898A",
    data_inizio="2026-03-01",
    data_fine="2026-06-07",
    tipo="nrt"
)

In [4]:
import pandas as pd


def df_misure_wide(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    # 1. Trova la colonna oraria
    if "Data_ora_dt" in d.columns:
        d["ora"] = pd.to_datetime(d["Data_ora_dt"], errors="coerce")
    elif "Data_ora" in d.columns:
        d["ora"] = (
            d["Data_ora"]
            .astype(str)
            .str.replace(r"\s+[+-]\d{2}:\d{2}$", "", regex=True)
        )
        d["ora"] = pd.to_datetime(
            d["ora"],
            format="%d-%m-%Y %H:%M:%S",
            errors="coerce"
        )
    elif "ora" in d.columns:
        d["ora"] = pd.to_datetime(d["ora"], errors="coerce")
    else:
        raise ValueError("Non trovo una colonna oraria: serve Data_ora_dt, Data_ora oppure ora.")

    # 2. Normalizza i nomi degli inquinanti
    d["inquinante_norm"] = (
        d["Inquinante"]
        .astype(str)
        .str.upper()
        .str.strip()
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    mapping = {
        "PM10": "pm10",
        "PM2.5": "pm25",
        "PM25": "pm25",
        "NO2": "no2",
        "CO" : 'co',
        "C6H6" : 'c6h6'
    }

    d["inquinante_col"] = d["inquinante_norm"].map(mapping)

    # 3. Tieni solo PM10, PM2.5 / PM25 e NO2
    d = d[d["inquinante_col"].notna()].copy()

    # 4. Assicura che Valore sia numerico
    d["Valore"] = (
        d["Valore"]
        .astype(str)
        .str.replace(",", ".", regex=False)
    )
    d["Valore"] = pd.to_numeric(d["Valore"], errors="coerce")

    # 5. Pivot: da long a wide
    out = (
        d.pivot_table(
            index=["Stazione", "ora"],
            columns="inquinante_col",
            values="Valore",
            aggfunc="first"
        )
        .reset_index()
    )

    # 6. Rinomina colonne
    out = out.rename(columns={
        "Stazione": "codice centralina"
    })

    # 7. Assicura che le colonne esistano anche se mancano dati
    for col in ["pm10", "pm25", "no2"]:
        if col not in out.columns:
            out[col] = pd.NA

    # 8. Ordina colonne
    out = out[[
        "codice centralina",
        "ora",
        "pm10",
        "pm25",
        "no2",
        'co',
        'c6h6'
    ]]

    # 9. Usa ora come indice
    out = out.sort_values("ora").set_index("ora")

    return out

In [5]:
df_wide = df_misure_wide(df)


In [6]:
df_wide

inquinante_col,codice centralina,pm10,pm25,no2,co,c6h6
ora,,,,,,
2026-03-01 00:00:00,IT0898A,55.9,43.9,77.67,1.17,3.136
2026-03-01 01:00:00,IT0898A,60.4,51.8,65.14,0.84,2.535
2026-03-01 02:00:00,IT0898A,54.1,43.6,54.83,0.77,2.031
2026-03-01 03:00:00,IT0898A,38.5,33.8,43.37,0.63,1.593
2026-03-01 04:00:00,IT0898A,32.5,31.5,40.44,0.57,1.300
...,...,...,...,...,...,...
2026-06-07 19:00:00,IT0898A,15.5,9.2,33.50,0.41,1.446
2026-06-07 20:00:00,IT0898A,14.9,11.5,35.41,0.38,1.316
2026-06-07 21:00:00,IT0898A,21.9,14.1,70.61,0.65,2.031
